# WikipediaML - ML-Powered Wikipedia Navigation

**Kaggle/Colab Setup Guide**

Bu notebook WikipediaML sistemini Kaggle veya Google Colab'da çalıştırmanızı sağlar.

## Önerilen Ayarlar:
- **Kaggle**: GPU P100 veya T4 (ücretsiz, haftada 30 saat)
- **Google Colab**: T4 GPU (ücretsiz)
- **Süre**: ~1-2 saat (1000 sayfa için)

## Adımlar:
1. Dependencies yükle
2. Wikipedia verisi indir (1000 sayfa)
3. Graph ve embeddings oluştur
4. MLP modelini eğit
5. Test et ve sonuçları indir

## 1. Setup ve Dependencies

In [ ]:
# Dependencies yükle
!pip install -q torch sentence-transformers faiss-cpu networkx scipy pandas tqdm python-igraph lxml beautifulsoup4

# GPU kontrolü
import torch
print(f"🖥️  Device: {'GPU ✅' if torch.cuda.is_available() else 'CPU ⚠️'}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Repository'yi Klonla veya Dosyaları Yükle

In [ ]:
# Seçenek 1: GitHub'dan klonla (eğer public repo ise)
# !git clone https://github.com/YOUR_USERNAME/WikipediaML.git
# %cd WikipediaML

# Seçenek 2: Kaggle Dataset olarak yükle
# 1. Tüm dosyaları zip'le: zip -r wikipediaml.zip .
# 2. Kaggle'a dataset olarak yükle
# 3. Notebook'ta dataset'i ekle

# Seçenek 3: Google Drive'dan yükle (Colab için)
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/WikipediaML

print("✅ Dosyalar hazır")

## 3. Wikipedia Verisi İndir (1000 sayfa - test için)

In [ ]:
!python3 scripts/download_wikipedia_dumps.py --limit 1000

## 4. Veriyi Parse Et ve Temizle

In [ ]:
!python3 scripts/parse_wikipedia_dumps.py

## 5. Graph Adjacency Matrix Oluştur

In [ ]:
!python3 scripts/build_adjacency_map.py

## 6. Embeddings ve FAISS Index Oluştur (GPU hızlandırmalı)

In [ ]:
!python3 scripts/build_embedding_index.py

## 7. Training Data Üret

In [ ]:
!python3 scripts/generate_training_data.py --num_samples 5000

## 8. MLP Scorer Modelini Eğit (GPU hızlandırmalı)

In [ ]:
!python3 scripts/train_mlp_scorer.py --epochs 20 --batch_size 512

## 9. Model Validasyonu

In [ ]:
!python3 scripts/validate_mlp_scorer.py

## 10. Benchmark Testleri

In [ ]:
!python3 scripts/benchmark_navigator.py

## 11. Manuel Test - Path Bulma

In [ ]:
import numpy as np
import pickle
from scipy.sparse import load_npz
from core.hybrid_scorer import HybridScorer
from core.beam_search import BeamSearchNavigator

# Load data
print("📥 Loading data...")
adjacency = load_npz('data/adjacency_map.npz')
with open('data/page_mappings.pkl', 'rb') as f:
    mappings = pickle.load(f)

embeddings = np.load('data/embeddings.npy')

# Create scorer and navigator
print("🧠 Initializing navigator...")
scorer = HybridScorer(
    embeddings=embeddings,
    mlp_model_path='models/mlp_scorer_best.pt'
)

navigator = BeamSearchNavigator(
    adjacency_matrix=adjacency,
    page_mappings=mappings,
    hybrid_scorer=scorer,
    beam_width=10
)

# Test paths
test_pairs = [
    ("Python", "Computer"),
    ("United States", "Europe"),
    ("Mathematics", "Physics")
]

print("\n🧪 Testing navigation...\n")
for start, target in test_pairs:
    print(f"🔍 {start} → {target}")
    result = navigator.search(start, target, max_depth=6)
    
    if result['path']:
        print(f"   ✅ Found: {' → '.join(result['path'])}")
        print(f"   Length: {len(result['path'])} steps")
        print(f"   Nodes explored: {result['nodes_explored']}")
    else:
        print(f"   ❌ No path found")
    print()

## 12. Sonuçları İndir

In [ ]:
# Önemli dosyaları zip'le
!zip -r wikipediaml_results.zip \
    data/adjacency_map.npz \
    data/embeddings.npy \
    data/faiss_index.bin \
    data/page_mappings.pkl \
    models/mlp_scorer_best.pt \
    models/training_history.json

print("✅ wikipediaml_results.zip hazır!")
print("📥 Kaggle: Output sekmesinden indir")
print("📥 Colab: files.download('wikipediaml_results.zip')")

## Tam Wikipedia Eğitimi İçin

Tüm Wikipedia'yı eğitmek için (6M+ sayfa):

```python
# 1. Limit olmadan indir (20GB+)
!python3 scripts/download_wikipedia_dumps.py

# 2. Yukarıdaki adımları tekrarla
# 3. Daha fazla epoch ve batch size kullan
!python3 scripts/train_mlp_scorer.py --epochs 50 --batch_size 1024
```

**Tahmini Süreler (GPU ile):**
- Download: 2-3 saat
- Parse: 4-6 saat
- Embeddings: 8-12 saat
- Training: 4-8 saat
- **Toplam: ~20-30 saat**